<a href="https://colab.research.google.com/github/aliza1800/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aliza1800/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions and reason codes

The queue ranks content for human review using a refresh score and model-based decline probability. Higher-ranked items are treated as higher-priority review candidates, not as automatic refresh decisions.

Each recommendation includes reason codes so a reviewer can see the main signals behind the priority. The reason codes describe observed patterns such as declining demand, low CTR on a visible page, older content, or weaker position.

The main action categories are:

1. **Refresh and review CTR** — prioritize when declining demand is observed together with low CTR on a relatively visible page.
2. **Refresh and review** — prioritize content showing multiple decline or freshness signals that warrant a broader content review.
3. **Review before action** — use when the model score is meaningful but the evidence is mixed, so a human should inspect the page and search intent first.

The ranking is a prioritization aid rather than a prediction of guaranteed future performance. The validated grouped-by-client results from ML-09 showed Precision@20 of 0.700 and Precision@50 of 0.740, so the queue should be treated as a shortlist for human review rather than as a set of certain decline cases.

In [34]:
from pathlib import Path
import pandas as pd

# Project paths
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "work").exists():
    PROJECT_ROOT = Path("/content/flyrank-ml-internship")

OUTPUT_DIR = PROJECT_ROOT / "work" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output directory:", OUTPUT_DIR)

Project root: /content/flyrank-ml-internship
Output directory: /content/flyrank-ml-internship/work/outputs


In [35]:
# Inspect the existing refresh queue sample

queue_path = PROJECT_ROOT / "outputs" / "refresh_queue_sample.csv"

queue = pd.read_csv(queue_path)

print("Shape:", queue.shape)
print("\nColumns:")
print(queue.columns.tolist())

print("\nFirst 5 rows:")
display(queue.head())

Shape: (200, 28)

Columns:
['final_rank', 'content_id', 'client_id', 'final_refresh_score', 'best_model_name', 'best_model_probability', 'baseline_refresh_score', 'confidence', 'suggested_action', 'final_reason_codes', 'is_declining_label', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'ctr', 'content_age_days', 'days_since_last_update', 'word_count', 'trend_direction', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']

First 5 rows:


,final_rank,content_id,client_id,final_refresh_score,best_model_name,best_model_probability,baseline_refresh_score,confidence,suggested_action,final_reason_codes,...,word_count,trend_direction,competition_level,content_type,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier
0,1,content_1f080331fa2b,client_3fdba35f04,81.636697,random_forest,0.782079,0.844481,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,...,1404.0,down,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,good,page_1
1,2,content_6aa43079fb0c,client_3fdba35f04,81.447656,random_forest,0.788105,0.825477,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1457.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1
2,3,content_d6570c51c9bd,client_3fdba35f04,81.430346,random_forest,0.847372,0.695884,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1362.0,down,MEDIUM,keyword article,informational,91-180,91-180,1000-2000,moderate,striking
3,4,content_72e800a9c214,client_3fdba35f04,81.034960,random_forest,0.774371,0.842545,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1371.0,down,MEDIUM,keyword article,commercial,91-180,91-180,1000-2000,good,page_1
4,5,content_e04eb9549989,client_3fdba35f04,80.873188,random_forest,0.814805,0.749468,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,...,1408.0,down,LOW,keyword article,informational,91-180,91-180,1000-2000,good,page_1


In [36]:
# Create a clean ranked action queue for ML-10

action_queue = queue.copy()

# Keep the most useful fields for human review and the paper
action_queue = action_queue[
    [
        "final_rank",
        "content_id",
        "final_refresh_score",
        "best_model_probability",
        "confidence",
        "suggested_action",
        "final_reason_codes",
        "impressions_90d",
        "avg_position",
        "ctr",
        "content_age_days",
        "days_since_last_update",
        "word_count",
        "trend_direction",
        "competition_level",
        "content_type",
        "main_intent",
        "age_tier",
        "freshness_tier",
        "position_tier",
    ]
].copy()

# Confirm that the queue remains ranked
action_queue = action_queue.sort_values(
    "final_rank",
    ascending=True
).reset_index(drop=True)

print("Action queue shape:", action_queue.shape)
print("\nTop 10 ranked actions:")

display(
    action_queue.head(10)[
        [
            "final_rank",
            "final_refresh_score",
            "best_model_probability",
            "confidence",
            "suggested_action",
            "final_reason_codes",
        ]
    ]
)

Action queue shape: (200, 20)

Top 10 ranked actions:


,final_rank,final_refresh_score,best_model_probability,confidence,suggested_action,final_reason_codes
0,1,81.636697,0.782079,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...
1,2,81.447656,0.788105,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...
2,3,81.430346,0.847372,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...
3,4,81.034960,0.774371,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...
4,5,80.873188,0.814805,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...
5,6,80.754770,0.795713,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...
6,7,80.632923,0.846245,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...
7,8,80.371236,0.834638,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...
8,9,80.362748,0.843092,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...
9,10,80.321757,0.843803,medium,refresh,declining_with_demand|model_decline_risk|visib...


### Archetype → action mapping

| Content archetype | Observed signals | Recommended human action |
|---|---|---|
| Declining + visible + low CTR | `trend_direction = down`, relatively strong visibility, low CTR | Review title/snippet alignment, search intent, and SERP competition before deciding on a refresh |
| Declining + aging content | Downward trend combined with high `content_age_days` or older age tier | Check whether facts, examples, and search intent have become outdated; refresh only if the review supports it |
| Declining + weaker position | Downward trend with weaker `avg_position` | Review topical coverage, intent match, internal linking, and competing results before changing content |
| Model-risk candidate | Higher model probability but mixed supporting signals | Human review required; inspect the page and underlying signals before taking action |
| Mixed or uncertain signals | Conflicting trend, CTR, position, or freshness signals | Do not act from the score alone; defer or review manually |

These mappings are decision-support rules. They do not prescribe automatic content changes.

### Decay / refresh insight

The earlier analysis showed that search-visibility decline was more common among older content. The observed decline rate was 69.4% for content aged 61–90 days, 64.8% for 91–120 days, 70.0% for 121–180 days, 61.4% for 181–270 days, 55.1% for 271–365 days, and 43.4% for content older than 365 days.

This pattern is directional and observational. It supports using content age and freshness as review signals, but it does not establish that aging itself causes search-visibility decline. Content should therefore be reviewed for outdated information, changed search intent, and competitive context before deciding whether a refresh is appropriate.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

This playbook is intended for content and SEO reviewers who need a ranked shortlist of pages to inspect for possible search-visibility decline.

The model output is used as decision support. A higher-ranked item means it should be considered earlier for human review; it does not mean that the page definitely needs a refresh.

The playbook can help reviewers:
- prioritize which content to inspect first;
- understand the main signals behind each recommendation;
- compare content patterns across age, visibility, CTR, and position;
- identify pages that may need further investigation.

### Limits

The model was validated using a grouped-by-client split in ML-09. Precision@20 was 0.700 and Precision@50 was 0.740 on that validation setup. These results support prioritization, but they do not establish that every ranked page will decline.

The analysis is observational and should not be interpreted as proof that any single feature causes search-visibility decline. The model uses signals such as content age, update recency, impressions, position, CTR, and word count, but these signals can reflect multiple underlying factors.

The playbook is not production automation. It should not publish, rewrite, delete, redirect, or otherwise change content without human review.

The recommendations may become less reliable when search behavior, site structure, content mix, tracking definitions, or the underlying data distribution changes.

### Cost / value thinking

Reviewer time is limited, so the ranked queue should be used to prioritize which pages are reviewed first rather than to decide which pages must be changed.

Higher-ranked pages can receive earlier human attention because they combine multiple observed signals associated with the review queue. However, model score alone does not establish business value. A page with high impressions, strong strategic importance, or clear business relevance may deserve review even when its model score is lower.

For lower-ranked or uncertain cases, monitoring or deferring review can be reasonable when the expected value of immediate review is unclear. Actual business value should be assessed by a human using page importance, search context, content quality, and later outcomes.

In [37]:
# Validation figures carried forward from ML-09

validation_metrics = {
    "grouped_by_client_precision_at_20": 0.700,
    "grouped_by_client_precision_at_50": 0.740,
}

print("ML-09 grouped-by-client validation:")
print(
    f"Precision@20: "
    f"{validation_metrics['grouped_by_client_precision_at_20']:.3f}"
)
print(
    f"Precision@50: "
    f"{validation_metrics['grouped_by_client_precision_at_50']:.3f}"
)

ML-09 grouped-by-client validation:
Precision@20: 0.700
Precision@50: 0.740


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review rules

Every ranked recommendation must be reviewed by a person before any content action is taken.

The reviewer should check:

1. **Search intent** — Does the page still match the intent of the query or topic it targets?
2. **Current content** — Are facts, examples, references, and recommendations still accurate and useful?
3. **SERP context** — What are competing pages currently offering, and has the search landscape changed?
4. **Performance signals** — Do impressions, CTR, position, and trend direction support the recommendation?
5. **Business context** — Is the page still relevant to the site's current goals and audience?
6. **Reason-code evidence** — Do the signals behind the recommendation make sense when the page is inspected?

A reviewer can choose to refresh, monitor, defer, or reject the recommendation. The model score should not override human judgment.

### No-go list

The following actions should **not** be automated by this playbook:

- Publishing or rewriting content automatically
- Deleting pages
- Creating redirects or changing URLs
- Changing canonical tags or other technical SEO settings
- Changing search intent or the target topic automatically
- Making claims about business impact from the model score alone
- Treating a high model probability as proof of future decline
- Applying the same recommendation to every page without contextual review
- Making changes when the underlying data is incomplete, stale, or inconsistent

The playbook is therefore a human-in-the-loop decision-support tool, not an autonomous content management system.

In [38]:
# Human-review gate check

review_required = True
automation_allowed = False

print("Human review required:", review_required)
print("Automatic content changes allowed:", automation_allowed)

Human review required: True
Automatic content changes allowed: False


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring

The playbook should be monitored for changes in recommendation quality and in the data used to produce the recommendations.

Useful monitoring signals include:

- **Precision@20 and Precision@50:** Recalculate these on a later labeled sample when actual outcomes become available.
- **Feature distribution:** Check whether important inputs such as content age, impressions, CTR, and average position have shifted substantially from the validation data.
- **Recommendation mix:** Monitor whether the queue suddenly contains very different proportions of actions, confidence levels, or content archetypes.
- **Missing or invalid values:** Check for unexpected missing values, new categories, or changes in metric definitions.
- **Outcome review:** Compare recommended pages with subsequent observed search-visibility outcomes after an appropriate observation period.

### Retrain triggers

A model review or retraining cycle should be considered when:

1. Precision@20 or Precision@50 shows a sustained decline on newly labeled data.
2. Important feature distributions change enough to make the original validation data less representative.
3. The content mix, search behavior, or tracking methodology changes materially.
4. New content types or client groups appear that were not represented adequately in the validation data.
5. The reason codes or recommendations repeatedly disagree with human reviewers.

These are review triggers rather than automatic retraining rules. A human should inspect the cause of the change before deciding whether to retrain or revise the playbook.

In [39]:
# Lightweight monitoring checks for the current queue

monitoring_checks = {
    "queue_rows": len(action_queue),
    "missing_values": int(action_queue.isna().sum().sum()),
    "unique_actions": int(action_queue["suggested_action"].nunique()),
    "unique_confidence_levels": int(action_queue["confidence"].nunique()),
}

print("Monitoring snapshot:")
for check, value in monitoring_checks.items():
    print(f"{check}: {value}")


Monitoring snapshot:
queue_rows: 200
missing_values: 0
unique_actions: 3
unique_confidence_levels: 2


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Paper exports

The ranked action queue is exported as a CSV so that the paper can reuse the same decision-support output without manually copying results from the notebook.

The exported queue contains the ranking, model probability, confidence, suggested action, reason codes, and the main content signals used for human review.

The queue is regenerated by the notebook rather than treated as a permanent production dataset. Client identifiers and other private information are excluded from the paper-facing export.

The export is intended for research reporting and review, not for direct automated content changes.

In [40]:
# Export the ML-10 ranked action queue

paper_queue = action_queue.drop(
    columns=["content_id"],
    errors="ignore"
).copy()

# Make sure the output directory exists
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

export_path = OUTPUT_DIR / "w07_ranked_action_queue.csv"

paper_queue.to_csv(export_path, index=False)

print("Export created:")
print(export_path)

print("\nExport shape:", paper_queue.shape)
print("\nExport columns:")
print(paper_queue.columns.tolist())

Export created:
/content/flyrank-ml-internship/work/outputs/w07_ranked_action_queue.csv

Export shape: (200, 19)

Export columns:
['final_rank', 'final_refresh_score', 'best_model_probability', 'confidence', 'suggested_action', 'final_reason_codes', 'impressions_90d', 'avg_position', 'ctr', 'content_age_days', 'days_since_last_update', 'word_count', 'trend_direction', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'position_tier']


In [41]:
# Verify that the exported queue can be read back

check_queue = pd.read_csv(export_path)

print("File exists:", export_path.exists())
print("Reloaded export shape:", check_queue.shape)

print("\nPrivate identifier check:")
print("client_id present:", "client_id" in check_queue.columns)
print("content_id present:", "content_id" in check_queue.columns)

print("\nTop 5 exported rows:")
display(check_queue.head())

File exists: True
Reloaded export shape: (200, 19)

Private identifier check:
client_id present: False
content_id present: False

Top 5 exported rows:


,final_rank,final_refresh_score,best_model_probability,confidence,suggested_action,final_reason_codes,impressions_90d,avg_position,ctr,content_age_days,days_since_last_update,word_count,trend_direction,competition_level,content_type,main_intent,age_tier,freshness_tier,position_tier
0,1,81.636697,0.782079,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,12834,6.8,0.05,165,104,1404.0,down,MEDIUM,keyword article,informational,91-180,91-180,page_1
1,2,81.447656,0.788105,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,8064,3.8,0.07,139,104,1457.0,down,LOW,keyword article,informational,91-180,91-180,page_1
2,3,81.430346,0.847372,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,2498,10.1,0.00,165,104,1362.0,down,MEDIUM,keyword article,informational,91-180,91-180,striking
3,4,81.034960,0.774371,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,13790,8.2,0.12,139,104,1371.0,down,MEDIUM,keyword article,commercial,91-180,91-180,page_1
4,5,80.873188,0.814805,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,3393,3.6,0.09,131,104,1408.0,down,LOW,keyword article,informational,91-180,91-180,page_1


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.